In [1]:
import pandas as pd

batches = pd.read_csv("../data/processed/batches.csv", parse_dates=["Date", "Start", "End"])

batches = batches.sort_values("Start").reset_index(drop=True)

batches[["Batch", "Date", "Product", "Operator", "Start", "Lost Time"]].head()

,Batch,Date,Product,Operator,Start,Lost Time
0,422111,2024-08-29,OR-600,Mac,2024-08-29 11:50:00,75.0
1,422112,2024-08-29,LE-600,Mac,2024-08-29 14:05:00,40.0
2,422113,2024-08-29,LE-600,Mac,2024-08-29 15:45:00,50.0
3,422114,2024-08-29,LE-600,Mac,2024-08-29 17:35:00,40.0
4,422115,2024-08-29,LE-600,Charlie,2024-08-29 19:15:00,24.0


In [2]:
batches["Prev Product"] = batches["Product"].shift(1)
batches["Changeover"] = (batches["Product"] != batches["Prev Product"]).astype(int)

batches["Prev Date"] = batches["Date"].shift(1)
batches["First of Day"] = (batches["Date"] != batches["Prev Date"]).astype(int)

batches["Start Hour"] = batches["Start"].dt.hour

batches[["Batch", "Product", "Prev Product", "Changeover", "First of Day", "Start Hour"]].head(10)

,Batch,Product,Prev Product,Changeover,First of Day,Start Hour
0,422111,OR-600,None,1,1,11
1,422112,LE-600,OR-600,1,0,14
2,422113,LE-600,LE-600,0,0,15
3,422114,LE-600,LE-600,0,0,17
4,422115,LE-600,LE-600,0,0,19
5,422116,LE-600,LE-600,0,0,20
6,422117,LE-600,LE-600,0,0,21
7,422118,CO-600,LE-600,1,1,4
8,422119,CO-600,CO-600,0,0,6
9,422120,CO-600,CO-600,0,0,7


In [3]:
features = ["Operator", "Size", "Changeover", "First of Day", "Start Hour"]

X = pd.get_dummies(batches[features], columns=["Operator", "Size"], dtype=int)
y = batches["Lost Time"]

print(X.shape)
X.head()

(31, 9)


,Changeover,First of Day,Start Hour,Operator_Charlie,Operator_Dee,Operator_Dennis,Operator_Mac,Size_2 L,Size_600 ml
0,1,1,11,0,0,0,1,0,1
1,1,0,14,0,0,0,1,0,1
2,0,0,15,0,0,0,1,0,1
3,0,0,17,0,0,0,1,0,1
4,0,0,19,1,0,0,0,0,1


In [4]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import mean_absolute_error

model = RandomForestRegressor(n_estimators=200, max_depth=3, random_state=42)
baseline = DummyRegressor(strategy="mean")

loo = LeaveOneOut()
model_preds = cross_val_predict(model, X, y, cv=loo)
baseline_preds = cross_val_predict(baseline, X, y, cv=loo)

print(f"Model MAE:    {mean_absolute_error(y, model_preds):.1f} minutes")
print(f"Baseline MAE: {mean_absolute_error(y, baseline_preds):.1f} minutes")

Model MAE:    21.8 minutes
Baseline MAE: 19.8 minutes


In [5]:
events = pd.read_csv("../data/processed/downtime_events.csv")

op_error_minutes = (
    events[events["Operator Error"] == "Yes"]
    .groupby("Batch")["Minutes"]
    .sum()
)

batches["Operator Error Minutes"] = batches["Batch"].map(op_error_minutes).fillna(0)
y_op = batches["Operator Error Minutes"]

model_preds = cross_val_predict(model, X, y_op, cv=loo)
baseline_preds = cross_val_predict(baseline, X, y_op, cv=loo)

print(f"Model MAE:    {mean_absolute_error(y_op, model_preds):.1f} minutes")
print(f"Baseline MAE: {mean_absolute_error(y_op, baseline_preds):.1f} minutes")

Model MAE:    17.1 minutes
Baseline MAE: 16.5 minutes
